In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()

    assert d_model % num_heads == 0
    self.d_model = d_model
    self.d_k = d_model // num_heads
    self.num_heads = num_heads

    self.W_q = nn.Linear(d_model, d_model)
    self.W_k = nn.Linear(d_model, d_model)
    self.W_v = nn.Linear(d_model, d_model)
    self.W_o = nn.Linear(d_model, d_model)

  def forward(self, x)  :
    batch, seq_len, _ = x.shape

    # Step-1: Create Q,K,V
    Q = self.W_q(x)
    K = self.W_k(x)
    V = self.W_v(x)

    # Step-2: Split into heads
    Q = Q.view(batch, seq_len, self.num_heads, self.d_k).transpose(1,2)
    K = K.view(batch, seq_len, self.num_heads, self.d_k).transpose(1,2)
    V = V.view(batch, seq_len, self.num_heads, self.d_k).transpose(1,2)

    # Step-3: Scores
    scores = torch.matmul(Q, K.transpose(-2,-1)) / (self.d_k ** 0.5)
    atten_weights = F.softmax(scores, dim=-1)
    headed_outputs = torch.matmul(atten_weights, V)

    # Step-4: concatenate heads
    headed_outputs = headed_outputs.transpose(1,2).contiguous()
    headed_outputs = headed_outputs.view(batch, seq_len, self.d_model)

    # Step-5: Output
    output = self.W_o(headed_outputs)
    return output, atten_weights

# Smaller Model

In [14]:
attn = MultiHeadAttention(d_model=24, num_heads=3)
x = torch.randn(2, 6, 24)
output, weights = attn(x)
print("Output shape:", output.shape)    # (2, 6, 24)
print("Weights shape:", weights.shape)  # (2, 3, 6, 6)
print("Row sums:", weights.sum(dim=-1)) # ~1.0

Output shape: torch.Size([2, 6, 24])
Weights shape: torch.Size([2, 3, 6, 6])
Row sums: tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]],
       grad_fn=<SumBackward1>)


# Bigger Model

In [15]:
attn = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(4, 10, 64)
output, weights = attn(x)
print("Output shape:", output.shape)
print("Weights shape:", weights.shape)


Output shape: torch.Size([4, 10, 64])
Weights shape: torch.Size([4, 8, 10, 10])


# Transfomer Model

In [16]:
attn = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 20, 512)
output, weights = attn(x)
print("Output:", output.shape)
print("Weights:", weights.shape)

Output: torch.Size([2, 20, 512])
Weights: torch.Size([2, 8, 20, 20])


# Bert Model

In [17]:
attn = MultiHeadAttention(d_model=768, num_heads=12)
x = torch.randn(1, 50, 768)
output, weights = attn(x)
print("Output:", output.shape)
print("Weights:", weights.shape)

Output: torch.Size([1, 50, 768])
Weights: torch.Size([1, 12, 50, 50])
